# CS336 Assignment 1 — Training Experiments

End-to-end pipeline using the components you built:
1. **Smoke test** the training loop on synthetic data (fast sanity check).
2. **Train a BPE tokenizer** on TinyStories.
3. **Encode** the corpus into a `uint16` `.npy` of token IDs.
4. **Train** the Transformer LM and watch train/val loss.
5. **Generate** text from the trained model.

To keep it laptop-friendly we use the TinyStories **validation** file (~22 MB) for everything and split it 90/10 into train/val. Scale up to the full train file once this runs clean.

In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))  # so `import cs336_basics` works from the notebook

import numpy as np
import torch

from cs336_basics.model import TransformerLM
from cs336_basics.nn_utils import cross_entropy
from cs336_basics.optimizer import AdamW, get_lr_cosine_schedule, gradient_clipping
from cs336_basics.data import get_batch
from cs336_basics.serialization import save_checkpoint, load_checkpoint
from cs336_basics.tokenizer import Tokenizer
from cs336_basics.train_bpe import train_bpe

# Pick the fastest device available
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device:", DEVICE)

DATA_DIR = "/Users/kuoweitseng/Desktop/cs336/data"
ART_DIR = "/Users/kuoweitseng/Desktop/cs336/artifacts"  # tokenizer + checkpoints go here
os.makedirs(ART_DIR, exist_ok=True)

device: mps


## 1. Smoke test the training loop

Before touching real data, confirm the loop runs and the loss moves. We train a tiny model on random tokens. The loss won't drop much (random data has no pattern), but **no crashes + a stable/slightly-decreasing loss** means the machinery is wired correctly.

In [2]:
def train_loop(
    data,                      # 1D int array (memmap or ndarray) of token IDs
    vocab_size,
    context_length,
    d_model,
    num_layers,
    num_heads,
    d_ff,
    rope_theta=10000.0,
    batch_size=32,
    max_steps=2000,
    max_lr=3e-4,
    min_lr=3e-5,
    warmup_steps=100,
    max_grad_norm=1.0,
    weight_decay=0.01,
    device=DEVICE,
    log_every=100,
    val_data=None,
    eval_every=500,
    eval_batches=20,
):
    model = TransformerLM(
        vocab_size, context_length, d_model, num_layers,
        num_heads, d_ff, rope_theta, device=device,
    ).to(device)
    opt = AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)

    def batch_loss(src):
        x, y = get_batch(src, batch_size, context_length, device)
        logits = model(x)
        return cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))

    @torch.no_grad()
    def eval_loss(src):
        model.eval()
        losses = [batch_loss(src).item() for _ in range(eval_batches)]
        model.train()
        return sum(losses) / len(losses)

    history = {"step": [], "train": [], "val": []}
    t0 = time.time()
    model.train()
    for step in range(max_steps):
        lr = get_lr_cosine_schedule(step, max_lr, min_lr, warmup_steps, max_steps)
        for g in opt.param_groups:
            g["lr"] = lr

        loss = batch_loss(data)
        opt.zero_grad()
        loss.backward()
        gradient_clipping(model.parameters(), max_grad_norm)
        opt.step()

        if step % log_every == 0:
            print(f"step {step:5d} | train {loss.item():.4f} | lr {lr:.2e} | {time.time()-t0:.0f}s")
        if val_data is not None and step % eval_every == 0:
            v = eval_loss(val_data)
            history["step"].append(step); history["train"].append(loss.item()); history["val"].append(v)
            print(f"    [eval] step {step} | val {v:.4f}")
    return model, opt, history

In [3]:
# Smoke test on synthetic random tokens
fake = np.random.randint(0, 1000, size=200_000).astype(np.uint16)
_ = train_loop(
    fake, vocab_size=1000, context_length=32,
    d_model=64, num_layers=2, num_heads=4, d_ff=128,
    batch_size=16, max_steps=100, log_every=20,
)

step     0 | train 6.9471 | lr 0.00e+00 | 1s
step    20 | train 6.9442 | lr 6.00e-05 | 2s
step    40 | train 6.9609 | lr 1.20e-04 | 2s
step    60 | train 6.9571 | lr 1.80e-04 | 2s
step    80 | train 6.9637 | lr 2.40e-04 | 3s


## 2. Train a BPE tokenizer on TinyStories

We train a 10K-vocab tokenizer on the validation file. With the un-optimized `train_bpe` this may take a few minutes (pre-tokenization is the bottleneck). It's cached to disk so you only do it once.

In [4]:
import pickle

VOCAB_PKL = os.path.join(ART_DIR, "ts_vocab.pkl")
MERGES_PKL = os.path.join(ART_DIR, "ts_merges.pkl")
SPECIAL = ["<|endoftext|>"]

if os.path.exists(VOCAB_PKL) and os.path.exists(MERGES_PKL):
    with open(VOCAB_PKL, "rb") as f: vocab = pickle.load(f)
    with open(MERGES_PKL, "rb") as f: merges = pickle.load(f)
    print("loaded cached tokenizer:", len(vocab), "vocab,", len(merges), "merges")
else:
    t0 = time.time()
    vocab, merges = train_bpe(
        input_path=os.path.join(DATA_DIR, "TinyStoriesV2-GPT4-valid.txt"),
        vocab_size=10000,
        special_tokens=SPECIAL,
    )
    print(f"trained tokenizer in {time.time()-t0:.0f}s: {len(vocab)} vocab, {len(merges)} merges")
    with open(VOCAB_PKL, "wb") as f: pickle.dump(vocab, f)
    with open(MERGES_PKL, "wb") as f: pickle.dump(merges, f)

tokenizer = Tokenizer(vocab, merges, special_tokens=SPECIAL)
eot_id = tokenizer.bytes_to_id["<|endoftext|>".encode("utf-8")]
print("longest token:", max(vocab.values(), key=len))

trained tokenizer in 113s: 10000 vocab, 9743 merges
longest token: b' accomplishment'


## 3. Encode the corpus to a `uint16` `.npy`

`encode_iterable` streams the file so we never load it all into memory. `uint16` because our vocab (10K) < 65536 = 2^16, so each ID fits in 2 bytes. Cached to disk.

In [5]:
TOKENS_NPY = os.path.join(ART_DIR, "ts_valid_tokens.npy")

if os.path.exists(TOKENS_NPY):
    tokens = np.load(TOKENS_NPY)
    print("loaded cached tokens:", tokens.shape)
else:
    t0 = time.time()
    ids = []
    with open(os.path.join(DATA_DIR, "TinyStoriesV2-GPT4-valid.txt"), encoding="utf-8") as f:
        for i, tid in enumerate(tokenizer.encode_iterable(f)):
            ids.append(tid)
            if i % 500_000 == 0 and i > 0:
                print(f"  {i:,} tokens | {time.time()-t0:.0f}s")
    tokens = np.array(ids, dtype=np.uint16)
    np.save(TOKENS_NPY, tokens)
    print(f"encoded {len(tokens):,} tokens in {time.time()-t0:.0f}s")

# 90/10 train/val split
n = len(tokens)
split = int(n * 0.9)
train_tokens = tokens[:split]
val_tokens = tokens[split:]
print(f"train {len(train_tokens):,} | val {len(val_tokens):,}")

KeyboardInterrupt: 

## 4. Train the model

A small config that trains in a few minutes on MPS/CPU. Increase `d_model`, `num_layers`, `max_steps` for better quality. Watch that **val loss** drops (real learning) instead of just train loss (which could be memorization).

In [ ]:
CONTEXT_LENGTH = 128
model, opt, history = train_loop(
    train_tokens,
    vocab_size=len(vocab),
    context_length=CONTEXT_LENGTH,
    d_model=256,
    num_layers=8,
    num_heads=8,
    d_ff=1024,
    batch_size=32,
    max_steps=2000,
    max_lr=3e-4,
    min_lr=3e-5,
    warmup_steps=100,
    device=DEVICE,
    log_every=100,
    val_data=val_tokens,
    eval_every=250,
)

save_checkpoint(model, opt, 2000, os.path.join(ART_DIR, "ts_model.pt"))
print("saved checkpoint")

In [ ]:
# Plot the loss curves
import matplotlib.pyplot as plt
plt.plot(history["step"], history["train"], label="train")
plt.plot(history["step"], history["val"], label="val")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("TinyStories LM")
plt.show()

## 5. Generate text

Autoregressive sampling with temperature + top-p (nucleus) sampling. This is the §6 decoding step. Feed a prompt, sample one token at a time, append, repeat — stopping at `<|endoftext|>` or a max length.

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, context_length, max_new_tokens=256,
             temperature=0.8, top_p=0.95, device=DEVICE, eot_id=None):
    model.eval()
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        logits = model(ids[:, -context_length:])[:, -1, :]  # last position's logits
        logits = logits / max(temperature, 1e-6)
        probs = torch.softmax(logits, dim=-1)
        # top-p (nucleus) filtering
        sp, si = torch.sort(probs, descending=True)
        cdf = torch.cumsum(sp, dim=-1)
        keep = cdf - sp <= top_p        # keep tokens until cumulative prob exceeds top_p
        sp = sp * keep
        sp = sp / sp.sum(dim=-1, keepdim=True)
        nxt = si.gather(-1, torch.multinomial(sp, 1))
        ids = torch.cat([ids, nxt], dim=1)
        if eot_id is not None and nxt.item() == eot_id:
            break
    return tokenizer.decode(ids[0].tolist())

print(generate(model, tokenizer, "Once upon a time", CONTEXT_LENGTH,
                max_new_tokens=200, temperature=0.8, top_p=0.95, eot_id=eot_id))

## Notes & next steps

- **Quality knobs**: bigger `d_model`/`num_layers`, more `max_steps`, longer `context_length`. Try `temperature` 0.6–1.0 when sampling.
- **Scale to full data**: point step 2/3 at `TinyStoriesV2-GPT4-train.txt` (2.2 GB). The `train_bpe` speed-up (incremental pair counts) becomes worth doing first. Load the `.npy` with `np.load(path, mmap_mode='r')` so you don't blow memory.
- **Perplexity** = `exp(val_loss)` — report it for the §7 deliverables.
- **Overfitting check**: if train loss drops but val loss rises, you're memorizing — add data or reduce model size.